# TSE Analysis 2026 — Ingestão de Logs da Urna (`fact_urna_logs`)

Este notebook implementa de forma didática e direta a ingestão da tabela **`fact_urna_logs`** a partir dos arquivos `.jez` (`logd.dat`) das **Eleições 2026**:
1. **Extração parcial via HTTP Range** de um arquivo `.jez` real do 1º turno de 2026 no Acre (`o03220ac0101500020088-log.jez`) diretamente do pacote oficial do TSE (`arqurnatot/bu_imgbu_logjez_rdv_vscmr_2026_1t_AC.zip`), baixando apenas ~68 KB em vez de 393 MB.
2. **Schema 1:1 (`fact_urna_logs` — Decisão A1)**: Preserva as colunas originais separadas por tabulação (`\t`) do `logd.dat` (`raw_timestamp`, `log_level`, `urna_serial`, `application`, `message_text` com números de 5+ dígitos mascarados como `<ID_REDACTED>`), omitindo `mac_code` para evitar ~35 GB de entropia hexadecimal aleatória e permitir alta taxa de compressão por dicionário em Parquet ZSTD.
3. **Inspeção de `HEAD` e `TAIL`** e validação de integridade física em **Polars**, **PyArrow (Parquet ZSTD)** e **DuckDB**.

In [1]:
import io
import json
import re
import struct
import tempfile
import zipfile
import zlib
from datetime import datetime
from pathlib import Path

import duckdb
import httpx
import polars as pl
import pyarrow.parquet as pq

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(16)
pl.Config.set_fmt_str_lengths(90)

polars.config.Config

In [2]:
# 1. Funções didáticas de leitura via HTTP Range e parsing 1:1 para fact_urna_logs
TSE_2026_AC_1T_URL = (
    "https://cdn.tse.jus.br/estatistica/sead/eleicoes/eleicoes2026/arqurnatot/"
    "bu_imgbu_logjez_rdv_vscmr_2026_1t_AC.zip"
)

JEZ_NAME_RE = re.compile(
    r"^o(?P<pleito>\d{5})(?P<uf>[a-z]{2})(?P<mun>\d{5})(?P<zona>\d{4})(?P<secao>\d{4})-log\.jez$",
    re.IGNORECASE,
)

FACT_URNA_LOGS_SCHEMA = pl.Schema({
    "election_year": pl.Int16,
    "turn": pl.Int8,
    "uf": pl.String,
    "cd_pleito": pl.Int32,
    "cd_municipio": pl.Int32,
    "nr_zona": pl.Int16,
    "nr_secao": pl.Int32,
    "source_jez_file": pl.String,
    "source_log_member": pl.String,
    "file_seq": pl.Int64,
    "raw_timestamp": pl.Datetime("us"),
    "log_level": pl.String,
    "urna_serial": pl.String,
    "application": pl.String,
    "message_text": pl.String,
})


def fetch_single_jez_from_remote_zip(zip_url: str, jez_index: int = 0) -> tuple[str, bytes]:
    """Lê o Central Directory no final do ZIP remoto via HTTP Range e baixa apenas 1 arquivo .jez."""
    with httpx.Client(timeout=30.0, follow_redirects=True) as client:
        total_size = int(client.head(zip_url).headers["content-length"])
        tail_len = min(total_size, 2 * 1024 * 1024)
        tail = client.get(
            zip_url, headers={"Range": f"bytes={total_size - tail_len}-{total_size - 1}"}
        ).content
        eocd_pos = tail.rfind(b"PK\x05\x06")
        cd_size, cd_offset = struct.unpack("<II", tail[eocd_pos + 12 : eocd_pos + 20])
        cd = client.get(
            zip_url, headers={"Range": f"bytes={cd_offset}-{cd_offset + cd_size - 1}"}
        ).content

        pos = 0
        found = []
        while pos + 46 <= len(cd):
            if cd[pos : pos + 4] != b"PK\x01\x02":
                break
            comp_method = struct.unpack("<H", cd[pos + 10 : pos + 12])[0]
            comp_size, _, fn_len, extra_len, comment_len = struct.unpack(
                "<IIHHH", cd[pos + 20 : pos + 34]
            )
            lhoff = struct.unpack("<I", cd[pos + 42 : pos + 46])[0]
            fname = cd[pos + 46 : pos + 46 + fn_len].decode("latin1")
            pos += 46 + fn_len + extra_len + comment_len
            if fname.lower().endswith(".jez"):
                found.append((fname, comp_method, comp_size, lhoff))
                if len(found) > jez_index:
                    break

        fname, comp_method, comp_size, lhoff = found[jez_index]
        raw_entry = client.get(
            zip_url, headers={"Range": f"bytes={lhoff}-{lhoff + 30 + len(fname) + 256 + comp_size}"}
        ).content
        fn_l, ex_l = struct.unpack("<HH", raw_entry[26:30])
        comp_data = raw_entry[30 + fn_l + ex_l : 30 + fn_l + ex_l + comp_size]
        jez_bytes = zlib.decompress(comp_data, -zlib.MAX_WBITS) if comp_method == 8 else comp_data
        return fname, jez_bytes


def sanitize_message_text(msg: str) -> str:
    """Mascara sequências numéricas de 5+ dígitos (ex.: título de mesário) preservando o texto 1:1."""
    return re.sub(r"\b\d{5,}\b", "<ID_REDACTED>", msg.strip())


def parse_jez_to_fact_urna_logs(
    jez_filename: str, jez_bytes: bytes, election_year: int = 2026, turn: int = 1
) -> tuple[list[str], pl.DataFrame]:
    """Extrai logd.dat em memória e converte 1:1 para o schema de fact_urna_logs (Decisão A1)."""
    m = JEZ_NAME_RE.match(Path(jez_filename).name)
    if not m:
        raise ValueError(f"Nome de arquivo .jez fora do padrão esperado: {jez_filename}")

    cd_pleito = int(m.group("pleito"))
    uf = m.group("uf").upper()
    cd_municipio = int(m.group("mun"))
    nr_zona = int(m.group("zona"))
    nr_secao = int(m.group("secao"))

    raw_lines_sanitized: list[str] = []
    rows: list[dict[str, object]] = []
    seq = 1

    with zipfile.ZipFile(io.BytesIO(jez_bytes), mode="r") as zf:
        members = sorted(
            [info.filename for info in zf.infolist() if not info.is_dir()],
            key=lambda name: (-(int(re.search(r"\.(\d+)$", name).group(1)) if re.search(r"\.(\d+)$", name) else 0), name),
        )
        for member_name in members:
            for raw_b in zf.read(member_name).splitlines():
                if not raw_b.strip():
                    continue
                line_str = raw_b.decode("latin1").rstrip("\r\n")
                parts = line_str.split("\t")
                ts = datetime.strptime(parts[0].strip(), "%d/%m/%Y %H:%M:%S")
                msg_clean = sanitize_message_text(parts[4])
                raw_lines_sanitized.append(
                    f"{parts[0].strip()}\t{parts[1].strip()}\t{parts[2].strip()}\t{parts[3].strip()}\t{msg_clean}"
                )
                rows.append({
                    "election_year": election_year,
                    "turn": turn,
                    "uf": uf,
                    "cd_pleito": cd_pleito,
                    "cd_municipio": cd_municipio,
                    "nr_zona": nr_zona,
                    "nr_secao": nr_secao,
                    "source_jez_file": Path(jez_filename).name,
                    "source_log_member": member_name,
                    "file_seq": seq,
                    "raw_timestamp": ts,
                    "log_level": parts[1].strip(),
                    "urna_serial": parts[2].strip(),
                    "application": parts[3].strip(),
                    "message_text": msg_clean,
                })
                seq += 1

    return raw_lines_sanitized, pl.DataFrame(rows, schema=FACT_URNA_LOGS_SCHEMA)

In [3]:
# 2. Extração do .jez real de 2026 e visualização de HEAD e TAIL do arquivo original (logd.dat)
jez_filename, jez_bytes = fetch_single_jez_from_remote_zip(TSE_2026_AC_1T_URL, jez_index=0)
raw_lines, df_urna_logs = parse_jez_to_fact_urna_logs(jez_filename, jez_bytes, election_year=2026, turn=1)

print(f"Arquivo .jez extraído: {jez_filename} ({len(jez_bytes):,} bytes comprimidos)")
print(f"Total de linhas no logd.dat: {len(raw_lines):,} | Linhas em fact_urna_logs: {df_urna_logs.height:,}\n")

print("=== HEAD (15 primeiras linhas de logd.dat — Decisão A1) ===")
for i, ln in enumerate(raw_lines[:15], start=1):
    print(f"{i:4d} | {ln}")

print("\n=== TAIL (15 últimas linhas de logd.dat — Decisão A1) ===")
for i, ln in enumerate(raw_lines[-15:], start=len(raw_lines) - 14):
    print(f"{i:4d} | {ln}")

Arquivo .jez extraído: o03220ac0101500020088-log.jez (68,091 bytes comprimidos)
Total de linhas no logd.dat: 3,714 | Linhas em fact_urna_logs: 3,714

=== HEAD (15 primeiras linhas de logd.dat — Decisão A1) ===
   1 | 23/09/2026 10:39:02	INFO	02263093	LOGD	Início das operações do logd
   2 | 23/09/2026 10:39:02	INFO	02263093	LOGD	Urna ligada em 23/09/2026 às 10:38:08
   3 | 23/09/2026 10:39:02	INFO	02263093	SCUE	Iniciando aplicação - 1º turno
   4 | 23/09/2026 10:39:02	INFO	02263093	SCUE	Versão da aplicação: 10.23.0.0 - Praia da Barra do Cahy
   5 | 23/09/2026 10:39:04	INFO	02263093	SCUE	Urna operando com rede elétrica
   6 | 23/09/2026 10:39:04	INFO	02263093	SCUE	Bateria interna com carga plena
   7 | 23/09/2026 10:39:10	INFO	02263093	SCUE	Tamanho da mídia interna: 3.7 GB
   8 | 23/09/2026 10:39:13	INFO	02263093	SCUE	Tamanho da memória: 3.7 GB
   9 | 23/09/2026 10:39:14	INFO	02263093	SCUE	Verificação de assinatura de aplicação por etapa [1] - [Pacote: /bin/avbin.vst] - [SUCESSO]
  10 |

In [4]:
# 3. Inspeção de HEAD (15) e TAIL (15) da tabela fact_urna_logs
print("=== HEAD (15) de fact_urna_logs ===")
print(df_urna_logs.head(15))

print("\n=== TAIL (15) de fact_urna_logs ===")
print(df_urna_logs.tail(15))

print("\n=== Amostra de eventos do aplicativo VOTA durante a votação (file_seq 706..722) ===")
print(
    df_urna_logs.filter(pl.col("file_seq").is_between(706, 722)).select(
        ["file_seq", "raw_timestamp", "log_level", "urna_serial", "application", "message_text"]
    )
)

=== HEAD (15) de fact_urna_logs ===
shape: (15, 15)
┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬────────┬───────┬───────┬───────┐
│ ele ┆ tur ┆ uf  ┆ cd_ ┆ cd_ ┆ nr_ ┆ nr_ ┆ sou ┆ sou ┆ fil ┆ raw ┆ log_le ┆ urna_ ┆ appli ┆ messa │
│ cti ┆ n   ┆ --- ┆ ple ┆ mun ┆ zon ┆ sec ┆ rce ┆ rce ┆ e_s ┆ _ti ┆ vel    ┆ seria ┆ catio ┆ ge_te │
│ on_ ┆ --- ┆ str ┆ ito ┆ ici ┆ a   ┆ ao  ┆ _je ┆ _lo ┆ eq  ┆ mes ┆ ---    ┆ l     ┆ n     ┆ xt    │
│ yea ┆ i8  ┆     ┆ --- ┆ pio ┆ --- ┆ --- ┆ z_f ┆ g_m ┆ --- ┆ tam ┆ str    ┆ ---   ┆ ---   ┆ ---   │
│ r   ┆     ┆     ┆ i32 ┆ --- ┆ i16 ┆ i32 ┆ ile ┆ emb ┆ i64 ┆ p   ┆        ┆ str   ┆ str   ┆ str   │
│ --- ┆     ┆     ┆     ┆ i32 ┆     ┆     ┆ --- ┆ er  ┆     ┆ --- ┆        ┆       ┆       ┆       │
│ i16 ┆     ┆     ┆     ┆     ┆     ┆     ┆ str ┆ --- ┆     ┆ dat ┆        ┆       ┆       ┆       │
│     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆ str ┆     ┆ eti ┆        ┆       ┆       ┆       │
│     ┆     ┆     ┆     ┆     ┆     ┆  

In [5]:
# 4. Persistência em Parquet ZSTD e verificação de integridade física via DuckDB
out_dir = Path(tempfile.mkdtemp(prefix="fact_urna_logs_"))
parquet_path = out_dir / "election_year=2026" / "turn=1" / "uf=AC" / "part-00000.parquet"
parquet_path.parent.mkdir(parents=True, exist_ok=True)
df_urna_logs.write_parquet(parquet_path, compression="zstd", compression_level=5)

pq_meta = pq.read_metadata(parquet_path)
parquet_size = parquet_path.stat().st_size
reduction_pct = (1.0 - parquet_size / len(jez_bytes)) * 100.0
print(
    f"Parquet ZSTD gravado: {parquet_path}\n"
    f"Tamanho Parquet ZSTD: {parquet_size:,} bytes vs {len(jez_bytes):,} bytes do .jez original "
    f"(redução de {reduction_pct:.1f}% sobre o arquivo já comprimido!)"
)

con = duckdb.connect()
cursor = con.execute(
    f"""
    SELECT
        application,
        log_level,
        COUNT(*) AS line_count,
        MIN(file_seq) AS min_seq,
        MAX(file_seq) AS max_seq,
        MIN(raw_timestamp) AS first_ts,
        MAX(raw_timestamp) AS last_ts
    FROM read_parquet('{parquet_path.as_posix()}')
    GROUP BY application, log_level
    ORDER BY line_count DESC
    """
)
cols = [d[0] for d in cursor.description]
summary_df = pl.DataFrame(cursor.fetchall(), schema=cols, orient="row")
print("\n=== Resumo de Integridade por Aplicativo e Nível de Log (DuckDB) ===")
print(summary_df)

# Validações de integridade da ingestão
assert df_urna_logs.height == len(raw_lines) == pq_meta.num_rows
assert df_urna_logs["file_seq"].to_list() == list(range(1, len(raw_lines) + 1))
assert df_urna_logs["raw_timestamp"].null_count() == 0

validation_summary = {
    "status": "SUCCESS",
    "table": "fact_urna_logs",
    "source_jez_file": jez_filename,
    "jez_compressed_bytes": len(jez_bytes),
    "parquet_zstd_bytes": parquet_size,
    "compression_reduction_pct": round(reduction_pct, 2),
    "total_rows": int(df_urna_logs.height),
    "null_timestamps": int(df_urna_logs["raw_timestamp"].null_count()),
}
Path("poc_validation_summary.json").write_text(json.dumps(validation_summary, indent=2), encoding="utf-8")
print("\nResumo de validação salvo em poc_validation_summary.json:")
print(json.dumps(validation_summary, indent=2))

Parquet ZSTD gravado: /tmp/fact_urna_logs_luzdslml/election_year=2026/turn=1/uf=AC/part-00000.parquet
Tamanho Parquet ZSTD: 33,768 bytes vs 68,091 bytes do .jez original (redução de 50.4% sobre o arquivo já comprimido!)

=== Resumo de Integridade por Aplicativo e Nível de Log (DuckDB) ===
shape: (8, 7)
┌─────────────┬───────────┬────────────┬─────────┬─────────┬───────────────┬───────────────────────┐
│ application ┆ log_level ┆ line_count ┆ min_seq ┆ max_seq ┆ first_ts      ┆ last_ts               │
│ ---         ┆ ---       ┆ ---        ┆ ---     ┆ ---     ┆ ---           ┆ ---                   │
│ str         ┆ str       ┆ i64        ┆ i64     ┆ i64     ┆ datetime[μs]  ┆ datetime[μs]          │
╞═════════════╪═══════════╪════════════╪═════════╪═════════╪═══════════════╪═══════════════════════╡
│ VOTA        ┆ INFO      ┆ 3161       ┆ 334     ┆ 3714    ┆ 2026-09-23    ┆ 2026-10-04 15:05:11   │
│             ┆           ┆            ┆         ┆         ┆ 10:58:10      ┆              